# Cost-of-Living Frontend API Notebook

This notebook is the frontend-facing analysis notebook for the cost-of-living dashboard. It follows `frontend_api_contract.md`: the notebook calls the HTTP API only, and the backend API reads/aggregates data from Elasticsearch.

Run the API locally or port-forward the Kubernetes API service before executing the notebook:

```bash
kubectl -n cost-living \
  port-forward svc/cost-living-platform-api 8010:80
```

Default local API base URL: `http://127.0.0.1:8000`.

## 1. Setup

The API contract says all chart APIs return JSON. Most chart endpoints return `{ rows: [...], meta: {...} }`; status endpoints may return plain objects.

In [ ]:
import os
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as patches

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None
    plt.style.use("ggplot")

API_BASE_URL = os.getenv("API_BASE_URL", "http://127.0.0.1:8000").rstrip("/")
# API_PREFIX = os.getenv("API_PREFIX", "/api/cost-living").rstrip("/")

DEFAULT_PARAMS = {
    "source_group": "all",
    "period": "month",
}

TOPICS = [
    "housing", "groceries", "fuel", "energy", "transport", "eating_out",
    "healthcare", "home_goods", "education", "inflation", "wages", "debt",
]

TOPIC_LABELS = {
    "housing": "Housing / Rent",
    "groceries": "Groceries",
    "fuel": "Fuel",
    "energy": "Electricity / Utilities",
    "transport": "Transport",
    "eating_out": "Eating Out",
    "healthcare": "Healthcare",
    "home_goods": "Home Goods",
    "education": "Education",
    "inflation": "Inflation",
    "wages": "Wages",
    "debt": "Debt",
}

def api_get(path, params=None, timeout=45):
    """Call the backend API. The backend is backed by Elasticsearch aggregations."""
    params = {k: v for k, v in (params or {}).items() if v not in (None, "")}
    url = f"{API_BASE_URL}{path}"
    response = requests.get(url, params=params, timeout=timeout)
    try:
        payload = response.json()
    except ValueError:
        response.raise_for_status()
        raise
    if not response.ok:
        raise RuntimeError(payload.get("detail", f"API request failed: {response.status_code}"))
    return payload

def rows_df(payload):
    """Convert either a rows response or a raw list response into a DataFrame."""
    if isinstance(payload, dict) and "rows" in payload:
        return pd.DataFrame(payload.get("rows", []))
    if isinstance(payload, list):
        return pd.DataFrame(payload)
    return pd.DataFrame([payload])

def normalize_columns(df):
    if df.empty:
        return df
    out = df.copy()
    if "category_label" not in out.columns and "cost_category" in out.columns:
        out["category_label"] = out["cost_category"].map(TOPIC_LABELS).fillna(out["cost_category"])
    if "complaint_count" not in out.columns and "document_count" in out.columns:
        out["complaint_count"] = out["document_count"]
    return out

def minmax(series):
    series = pd.to_numeric(series, errors="coerce")
    span = series.max() - series.min()
    if pd.isna(span) or span == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / span

def show_empty(title, message="No rows returned for the selected filters."):
    print(f"{title}: {message}")

TREEMAP_COLORS = [
    "#2F5D62", "#5E8C6A", "#D6A85B", "#C66B5C", "#6C7A9C", "#8A6F8E",
    "#7EA8A2", "#A88F6A", "#B7A7A0", "#4F6F8F", "#9A7B4F", "#6F8F72",
]
BAR_COLOR = "#2F5D62"
BAR_COLOR_ALT = "#4F6F8F"
BAR_COLOR_MUTED = "#7EA8A2"
NEGATIVE_COLOR = "#B45F5F"
POSITIVE_COLOR = "#5E8C6A"
NEUTRAL_COLOR = "#B7A7A0"
SENTIMENT_COLORS = {
    "positive_ratio": POSITIVE_COLOR,
    "neutral_ratio": NEUTRAL_COLOR,
    "negative_ratio": NEGATIVE_COLOR,
}
SENTIMENT_LABEL_COLORS = {
    "positive": POSITIVE_COLOR,
    "neutral": NEUTRAL_COLOR,
    "negative": NEGATIVE_COLOR,
}
PIPELINE_COLORS = {
    "Processed": "#2F5D62",
    "Discarded": "#B7A7A0",
    "Unprocessed": "#D6A85B",
    "Failed": "#B45F5F",
}
PLATFORM_COLORS = {
    "gdelt": "#4F6F8F",
    "mastodon": "#5E8C6A",
    "bluesky": "#D6A85B",
}
CATEGORY_COLORS = dict(zip(TOPICS, TREEMAP_COLORS))
CATEGORY_LABEL_COLORS = {TOPIC_LABELS[k]: v for k, v in CATEGORY_COLORS.items()}

def color_for_category(category=None, label=None):
    if category in CATEGORY_COLORS:
        return CATEGORY_COLORS[category]
    if label in CATEGORY_LABEL_COLORS:
        return CATEGORY_LABEL_COLORS[label]
    return BAR_COLOR_MUTED

def category_colors(df):
    if df.empty:
        return []
    if "cost_category" in df.columns:
        return [color_for_category(category=x) for x in df["cost_category"]]
    if "category_label" in df.columns:
        return [color_for_category(label=x) for x in df["category_label"]]
    return [BAR_COLOR_MUTED] * len(df)

def category_colors_for_labels(labels):
    return [color_for_category(label=label) for label in labels]

def compact_number(value):
    value = 0 if pd.isna(value) else float(value)
    if abs(value) >= 1_000_000:
        return f"{value / 1_000_000:.1f}M"
    if abs(value) >= 1_000:
        return f"{value / 1_000:.1f}K"
    return f"{value:.0f}"

def palette_for_index(index, color_map, fallback=BAR_COLOR_MUTED):
    return [color_map.get(str(item).lower(), color_map.get(str(item), fallback)) for item in index]

def plot_donut(series, title, colors=None, center_text=None, figsize=(6.4, 4.8)):
    data = pd.Series(series).dropna()
    data = data[pd.to_numeric(data, errors="coerce") > 0]
    if data.empty:
        show_empty(title)
        return
    colors = colors or TREEMAP_COLORS[:len(data)]
    fig, ax = plt.subplots(figsize=figsize)
    wedges, texts, autotexts = ax.pie(
        data.values,
        labels=data.index,
        colors=colors,
        startangle=90,
        counterclock=False,
        wedgeprops={"width": 0.42, "edgecolor": "white", "linewidth": 2},
        autopct=lambda pct: f"{pct:.1f}%" if pct >= 3 else "",
        pctdistance=0.78,
        labeldistance=1.05,
        textprops={"fontsize": 10, "color": "#333333"},
    )
    for text in autotexts:
        text.set_color("white")
        text.set_weight("bold")
        text.set_fontsize(9)
    if center_text:
        ax.text(0, 0, center_text, ha="center", va="center", fontsize=13, weight="bold", color="#333333")
    ax.set_title(title, pad=12)
    plt.tight_layout()
    plt.show()

def plot_stacked_status_bar(series, title):
    data = pd.Series(series).dropna()
    data = data[data > 0]
    if data.empty:
        show_empty(title)
        return
    total = data.sum()
    fig, ax = plt.subplots(figsize=(10, 1.8))
    left = 0
    for label, value in data.items():
        color = PIPELINE_COLORS.get(label, BAR_COLOR_MUTED)
        ax.barh([0], [value], left=left, height=0.48, color=color, label=f"{label}: {compact_number(value)}")
        if value / total >= 0.06:
            ax.text(left + value / 2, 0, f"{value / total:.1%}", ha="center", va="center", color="white", weight="bold", fontsize=10)
        left += value
    ax.set_xlim(0, total)
    ax.set_yticks([])
    ax.set_xlabel("Documents")
    ax.set_title(title)
    ax.legend(ncol=min(len(data), 4), bbox_to_anchor=(0.5, -0.45), loc="upper center", frameon=False)
    plt.tight_layout()
    plt.show()

plt.rcParams.update({
    "axes.edgecolor": "#D8D4CC",
    "axes.labelcolor": "#3A3A3A",
    "axes.titleweight": "semibold",
    "axes.titlesize": 13,
    "figure.facecolor": "white",
    "grid.color": "#ECE8E0",
    "grid.linewidth": 0.8,
})

def plot_treemap(df, label_col, value_col, title, figsize=(11, 6), colors=None):
    """Simple slice-and-dice treemap without extra notebook dependencies."""
    data = df[[label_col, value_col]].dropna().copy()
    data[value_col] = pd.to_numeric(data[value_col], errors="coerce")
    data = data[data[value_col] > 0].sort_values(value_col, ascending=False).reset_index(drop=True)
    if data.empty:
        show_empty(title)
        return

    total = data[value_col].sum()
    colors = colors or TREEMAP_COLORS
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    x, y, w, h = 0.0, 0.0, 1.0, 1.0
    horizontal = True
    for i, row in enumerate(data.itertuples(index=False)):
        label = getattr(row, label_col)
        value = float(getattr(row, value_col))
        share = value / total
        remaining = data[value_col].iloc[i:].sum()
        area = w * h * value / max(remaining, 1e-9)
        if horizontal:
            rect_w = area / h if h else 0
            rect = (x, y, rect_w, h)
            x += rect_w
            w -= rect_w
        else:
            rect_h = area / w if w else 0
            rect = (x, y, w, rect_h)
            y += rect_h
            h -= rect_h
        horizontal = not horizontal

        rx, ry, rw, rh = rect
        ax.add_patch(patches.Rectangle((rx, ry), rw, rh, facecolor=colors[i % len(colors)], edgecolor="white", linewidth=2))
        pct = share * 100
        if rw * rh > 0.035:
            ax.text(rx + rw / 2, ry + rh / 2, f"{label}\n{pct:.1f}%", ha="center", va="center", fontsize=10, color="white", weight="bold", wrap=True)
        elif rw * rh > 0.015:
            ax.text(rx + rw / 2, ry + rh / 2, f"{pct:.1f}%", ha="center", va="center", fontsize=9, color="white", weight="bold")

    ax.set_title(title, pad=12)
    plt.tight_layout()
    plt.show()

print(f"Using API_BASE_URL = {API_BASE_URL}")

## 2. Health Check

This checks whether the API and ES-backed indices are reachable.

In [ ]:
health = api_get("/api/health")
pd.Series(health).to_frame("value")

## 3. Pipeline Status

Charts covered: Pipeline Status Cards/Table and Raw vs Processed Documents.

In [ ]:
status = api_get("/api/pipeline/status")

status_metrics = [
    "raw_documents", "processed_documents", "unprocessed_documents",
    "discarded_documents", "failed_documents", "official_indicators",
    "last_harvest_at", "last_processed_at", "latest_official_period",
]
status_labels = {
    "raw_documents": "Raw documents",
    "processed_documents": "Processed documents",
    "unprocessed_documents": "Unprocessed documents",
    "discarded_documents": "Discarded documents",
    "failed_documents": "Failed documents",
    "official_indicators": "Official CPI indicators",
    "last_harvest_at": "Last harvest at",
    "last_processed_at": "Last processed at",
    "latest_official_period": "Latest official period",
}
status_table = pd.DataFrame(
    [{"metric": status_labels.get(key, key), "value": status.get(key)} for key in status_metrics if key in status]
)
display(status_table)

official_indicators = int(status.get("official_indicators", 0) or 0)
print(f"Official CPI indicators count: {official_indicators:,}")

processed = int(status.get("processed_documents", 0) or 0)
discarded = int(status.get("discarded_documents", 0) or 0)
unprocessed = int(status.get("unprocessed_documents", 0) or 0)
failed = int(status.get("failed_documents", 0) or 0)
raw_total = int(status.get("raw_documents", processed + discarded + unprocessed + failed) or 0)

processing_status = pd.Series({
    "Processed": processed,
    "Discarded": discarded,
    "Unprocessed": unprocessed,
    "Failed": failed,
})
processing_rate = processed / raw_total if raw_total else 0

plot_donut(
    processing_status,
    "Pipeline Processing Status",
    colors=[PIPELINE_COLORS[label] for label in processing_status.index],
    center_text=f"{processing_rate:.1%}\nprocessed",
    figsize=(6.2, 5.2),
)
plot_stacked_status_bar(processing_status, f"Raw Document Outcomes (total {raw_total:,})")

sources_df = pd.DataFrame(status.get("sources", []))
if not sources_df.empty:
    display(sources_df)


## 4. Overview

Charts covered: Documents by Platform and Total Documents / Complaints / Sentiment Overview.

In [ ]:
overview = api_get("/api/stats/overview", {"source_group": DEFAULT_PARAMS["source_group"]})

total_documents = int(overview.get("total_documents", 0) or 0)
total_complaints = int(overview.get("total_complaints", 0) or 0)
negative_documents = int(overview.get("negative_documents", 0) or 0)
non_negative_documents = max(total_documents - negative_documents, 0)

overview_metrics = pd.DataFrame([
    {"metric": "Total Documents", "value": total_documents},
    {"metric": "Total Complaints", "value": total_complaints},
    {"metric": "Negative Documents", "value": negative_documents},
    {"metric": "Negative Share", "value": f"{(negative_documents / total_documents):.1%}" if total_documents else "0.0%"},
    {"metric": "Date Start", "value": (overview.get("date_range") or {}).get("start")},
    {"metric": "Date End", "value": (overview.get("date_range") or {}).get("end")},
])
display(overview_metrics)

platforms = pd.Series(overview.get("platforms", {}), name="documents").sort_values(ascending=False)
if len(platforms):
    plot_donut(
        platforms,
        "Documents by Platform",
        colors=palette_for_index(platforms.index, PLATFORM_COLORS, fallback=BAR_COLOR_MUTED),
        center_text=f"{compact_number(platforms.sum())}\ndocs",
        figsize=(6.5, 5.2),
    )
else:
    show_empty("Documents by Platform")

sentiment = pd.Series(overview.get("sentiment", {}), name="documents")
if len(sentiment):
    sentiment = sentiment.reindex([x for x in ["positive", "neutral", "negative"] if x in sentiment.index]).dropna()
    plot_donut(
        sentiment,
        "Sentiment Distribution",
        colors=palette_for_index(sentiment.index, SENTIMENT_LABEL_COLORS, fallback=BAR_COLOR_MUTED),
        center_text=f"{compact_number(sentiment.sum())}\ntexts",
        figsize=(6.5, 5.2),
    )
elif total_documents:
    negative_split = pd.Series({"Non-negative": non_negative_documents, "Negative": negative_documents})
    plot_donut(
        negative_split,
        "Negative vs Non-negative Documents",
        colors=[NEUTRAL_COLOR, NEGATIVE_COLOR],
        center_text=f"{negative_documents / total_documents:.1%}\nnegative",
        figsize=(6.5, 5.2),
    )


## 5. Documents Over Time

Charts covered: Documents Ingested/Created Over Time and Documents by Platform Over Time. Use `period=day` for recent pipeline monitoring, or `period=month` for stable analysis.

In [ ]:
documents_df = normalize_columns(rows_df(api_get(
    "/api/trends/documents",
    {"source_group": DEFAULT_PARAMS["source_group"], "period": DEFAULT_PARAMS["period"], "time_field": "created_at"},
)))

if documents_df.empty:
    show_empty("Documents Over Time")
else:
    display(documents_df.head())
    total_by_period = documents_df.groupby("period", as_index=False)["document_count"].sum()
    ax = total_by_period.plot(x="period", y="document_count", figsize=(10, 4), marker="o", legend=False)
    ax.set_title("Documents Over Time")
    ax.set_xlabel("Period")
    ax.set_ylabel("Documents")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    platform_pivot = documents_df.pivot_table(index="period", columns="platform", values="document_count", aggfunc="sum", fill_value=0)
    ax = platform_pivot.plot(figsize=(10, 4), marker="o")
    ax.set_title("Documents Over Time by Platform")
    ax.set_xlabel("Period")
    ax.set_ylabel("Documents")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 6. Category Counts

Charts covered: Complaint Count by Cost-of-Living Category and Category Share/Treemap-style view.

In [ ]:
counts_df = normalize_columns(rows_df(api_get(
    "/api/categories/counts",
    {"source_group": DEFAULT_PARAMS["source_group"]},
))).sort_values("complaint_count", ascending=False)

if counts_df.empty:
    show_empty("Category Counts")
else:
    display(counts_df)
    ax = counts_df.plot(kind="barh", x="category_label", y="complaint_count", legend=False, figsize=(9, 6), color=category_colors(counts_df))
    ax.invert_yaxis()
    ax.set_title("Complaint Count by Cost-of-Living Category")
    ax.set_xlabel("Complaints / Documents")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

    share_value_col = "percentage" if "percentage" in counts_df.columns and counts_df["percentage"].fillna(0).sum() > 0 else "complaint_count"
    plot_treemap(counts_df, "category_label", share_value_col, "Category Share Treemap", colors=category_colors(counts_df))

## 7. Sentiment by Category

Charts covered: Average Sentiment by Category, Negative Ratio by Category, and Sentiment Composition by Category.

In [ ]:
sentiment_df = normalize_columns(rows_df(api_get(
    "/api/categories/sentiment",
    {"source_group": DEFAULT_PARAMS["source_group"]},
)))

if sentiment_df.empty:
    show_empty("Sentiment by Category")
else:
    display(sentiment_df.sort_values("negative_ratio", ascending=False))

    avg_df = sentiment_df.sort_values("avg_sentiment")
    ax = avg_df.plot(kind="barh", x="category_label", y="avg_sentiment", legend=False, figsize=(9, 6), color=category_colors(avg_df))
    ax.set_title("Average Sentiment by Category")
    ax.set_xlabel("Average Sentiment")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

    neg_df = sentiment_df.sort_values("negative_ratio", ascending=False).copy()
    neg_df["negative_percent"] = neg_df["negative_ratio"] * 100
    ax = neg_df.plot(kind="barh", x="category_label", y="negative_percent", legend=False, figsize=(9, 6), color=category_colors(neg_df))
    ax.invert_yaxis()
    ax.set_title("Negative Ratio by Category")
    ax.set_xlabel("Negative documents (%)")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

    ratio_cols = [c for c in ["positive_ratio", "neutral_ratio", "negative_ratio"] if c in sentiment_df.columns]
    comp = sentiment_df.set_index("category_label")[ratio_cols] * 100
    ax = comp.plot(kind="bar", stacked=True, figsize=(11, 5), color=[SENTIMENT_COLORS[c] for c in ratio_cols])
    ax.set_title("Sentiment Composition by Category")
    ax.set_xlabel("Category")
    ax.set_ylabel("Share (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.ticker import FuncFormatter

counts_df = normalize_columns(rows_df(api_get("/api/categories/counts")))
sentiment_df = normalize_columns(rows_df(api_get("/api/categories/sentiment")))

volume_sentiment_df = counts_df.merge(
    sentiment_df,
    on=["cost_category", "category_label"],
    how="inner",
    suffixes=("_count", "_sentiment")
)

display(volume_sentiment_df.head())
print(volume_sentiment_df.columns)

possible_volume_cols = [
    "document_count_count",
    "complaint_count_count",
    "document_count_x",
    "complaint_count_x",
    "document_count",
    "complaint_count"
]

x_col = None
for col in possible_volume_cols:
    if col in volume_sentiment_df.columns:
        x_col = col
        break

if x_col is None:
    raise ValueError(
        "No topic volume column found. Available columns are: "
        + ", ".join(volume_sentiment_df.columns)
    )

if "negative_ratio" not in volume_sentiment_df.columns:
    raise ValueError(
        "negative_ratio column not found. Available columns are: "
        + ", ".join(volume_sentiment_df.columns)
    )

volume_sentiment_df[x_col] = pd.to_numeric(volume_sentiment_df[x_col], errors="coerce")
volume_sentiment_df["negative_ratio"] = pd.to_numeric(
    volume_sentiment_df["negative_ratio"],
    errors="coerce"
)

plot_df = volume_sentiment_df.dropna(
    subset=[x_col, "negative_ratio", "category_label"]
).copy()

if plot_df.empty:
    raise ValueError("No valid data available for plotting.")

max_volume = plot_df[x_col].max()
if max_volume == 0:
    bubble_sizes = [300] * len(plot_df)
else:
    bubble_sizes = (plot_df[x_col] / max_volume) * 1600 + 80

fig, ax = plt.subplots(figsize=(12, 7))

fig.patch.set_facecolor("white")
ax.set_facecolor("#E5ECF6")

ax.scatter(
    plot_df[x_col],
    plot_df["negative_ratio"],
    s=bubble_sizes,
    alpha=0.65,
    color="#636EFA",
    edgecolors="white",
    linewidth=1.2
)

for _, row in plot_df.iterrows():
    ax.annotate(
        str(row["category_label"]),
        xy=(row[x_col], row["negative_ratio"]),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=11,
        color="#2A3F5F"
    )

ax.set_title(
    "Topic Volume vs Negative Sentiment",
    fontsize=20,
    color="#2A3F5F",
    pad=22
)

ax.set_xlabel(
    "Topic Volume",
    fontsize=14,
    color="#2A3F5F",
    labelpad=12
)

ax.set_ylabel(
    "Negative Ratio",
    fontsize=14,
    color="#2A3F5F",
    labelpad=12
)

def format_thousands(x, pos):
    if x >= 1000:
        return f"{int(x/1000)}K"
    return str(int(x))

ax.xaxis.set_major_formatter(FuncFormatter(format_thousands))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

ax.grid(True, color="white", linewidth=1.2)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="both", labelsize=12, colors="#2A3F5F")

x_min = plot_df[x_col].min()
x_max = plot_df[x_col].max()
y_min = plot_df["negative_ratio"].min()
y_max = plot_df["negative_ratio"].max()

x_padding = (x_max - x_min) * 0.12 if x_max != x_min else 100
y_padding = (y_max - y_min) * 0.18 if y_max != y_min else 0.05

ax.set_xlim(max(0, x_min - x_padding), x_max + x_padding)
ax.set_ylim(max(0, y_min - y_padding), min(1, y_max + y_padding))

plt.tight_layout()
plt.show()

## 8. Platform x Category

Charts covered: Platform x Category Heatmap, Normalised Heatmap, and Average Sentiment by Platform and Category.

In [ ]:
platform_category_df = normalize_columns(rows_df(api_get("/api/platforms/categories")))

if platform_category_df.empty:
    show_empty("Platform x Category")
else:
    display(platform_category_df.head())
    count_matrix = platform_category_df.pivot_table(index="platform", columns="category_label", values="count", aggfunc="sum", fill_value=0)
    norm_matrix = platform_category_df.pivot_table(index="platform", columns="category_label", values="percentage_within_platform", aggfunc="sum", fill_value=0) * 100
    sent_matrix = platform_category_df.pivot_table(index="platform", columns="category_label", values="avg_sentiment", aggfunc="mean")

    if sns:
        plt.figure(figsize=(12, 4))
        sns.heatmap(count_matrix, cmap="YlGnBu", annot=False)
        plt.title("Platform x Category Count Heatmap")
        plt.xlabel("Category")
        plt.ylabel("Platform")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 4))
        sns.heatmap(norm_matrix, cmap="BuGn", annot=False)
        plt.title("Platform x Category Normalised Heatmap (%)")
        plt.xlabel("Category")
        plt.ylabel("Platform")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 4))
        sns.heatmap(sent_matrix, cmap="coolwarm", center=0, annot=False)
        plt.title("Average Sentiment by Platform and Category")
        plt.xlabel("Category")
        plt.ylabel("Platform")
        plt.tight_layout()
        plt.show()
    else:
        display(count_matrix)
        display(norm_matrix)
        display(sent_matrix)

## 9. Category Trends

Charts covered: Recent Complaint Trend and Complaint Trend by Category.

In [ ]:
trend_df = normalize_columns(rows_df(api_get(
    "/api/trends/categories",
    {"source_group": DEFAULT_PARAMS["source_group"], "period": DEFAULT_PARAMS["period"]},
)))

if trend_df.empty:
    show_empty("Category Trends")
else:
    display(trend_df.head())
    total_trend = trend_df.groupby("period", as_index=False)["complaint_count"].sum()
    ax = total_trend.plot(x="period", y="complaint_count", marker="o", legend=False, figsize=(10, 4))
    ax.set_title("Recent Complaint Trend")
    ax.set_xlabel("Period")
    ax.set_ylabel("Complaints / Documents")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    top_categories = counts_df.head(6)["cost_category"].tolist() if "counts_df" in globals() and not counts_df.empty else trend_df["cost_category"].dropna().unique()[:6]
    trend_top = trend_df[trend_df["cost_category"].isin(top_categories)]
    pivot = trend_top.pivot_table(index="period", columns="category_label", values="complaint_count", aggfunc="sum", fill_value=0)
    ax = pivot.plot(figsize=(11, 5), marker="o", color=category_colors_for_labels(pivot.columns))
    ax.set_title("Complaint Trend by Category")
    ax.set_xlabel("Period")
    ax.set_ylabel("Complaints / Documents")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 10. Sentiment Trends

Charts covered: Negative Sentiment Ratio Over Time and Average Sentiment Over Time.

In [ ]:
sentiment_trend_df = normalize_columns(rows_df(api_get(
    "/api/trends/sentiment",
    {"source_group": DEFAULT_PARAMS["source_group"], "period": DEFAULT_PARAMS["period"]},
)))

if sentiment_trend_df.empty:
    show_empty("Sentiment Trends")
else:
    top_categories = counts_df.head(6)["cost_category"].tolist() if "counts_df" in globals() and not counts_df.empty else sentiment_trend_df["cost_category"].dropna().unique()[:6]
    sentiment_top = sentiment_trend_df[sentiment_trend_df["cost_category"].isin(top_categories)].copy()

    neg_pivot = sentiment_top.pivot_table(index="period", columns="category_label", values="negative_ratio", aggfunc="mean") * 100
    ax = neg_pivot.plot(figsize=(11, 5), marker="o", color=category_colors_for_labels(neg_pivot.columns))
    ax.set_title("Negative Sentiment Ratio Over Time")
    ax.set_xlabel("Period")
    ax.set_ylabel("Negative documents (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    avg_pivot = sentiment_top.pivot_table(index="period", columns="category_label", values="avg_sentiment", aggfunc="mean")
    ax = avg_pivot.plot(figsize=(11, 5), marker="o", color=category_colors_for_labels(avg_pivot.columns))
    ax.set_title("Average Sentiment Over Time")
    ax.set_xlabel("Period")
    ax.set_ylabel("Average sentiment")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 11. Official CPI Comparison

Charts covered: Official Indicator vs Complaint Volume, Official Indicator vs Negative Sentiment, and Correlation Heatmap.

The contract says this endpoint is monthly only. Treat correlations as contextual evidence when text coverage is short.

In [ ]:
comparison_df = normalize_columns(rows_df(api_get(
    "/api/official/comparison",
    {"source_group": DEFAULT_PARAMS["source_group"], "topic": "housing,groceries,fuel,energy,transport"},
)))

if comparison_df.empty:
    show_empty("Official CPI Comparison")
else:
    display(comparison_df.head())
    focus_category = "groceries" if "groceries" in comparison_df["cost_category"].values else comparison_df["cost_category"].dropna().iloc[0]
    focus = comparison_df[comparison_df["cost_category"] == focus_category].sort_values("period").copy()
    focus["official_norm"] = minmax(focus["official_value"])
    focus["complaints_norm"] = minmax(focus["complaint_count"])
    focus["negative_norm"] = minmax(focus["negative_ratio"])

    ax = focus.plot(x="period", y=["official_norm", "complaints_norm"], marker="o", figsize=(10, 4))
    ax.set_title(f"Official CPI vs Complaint Volume ({TOPIC_LABELS.get(focus_category, focus_category)})")
    ax.set_xlabel("Period")
    ax.set_ylabel("Normalised value")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    ax = focus.plot(x="period", y=["official_norm", "negative_norm"], marker="o", figsize=(10, 4))
    ax.set_title(f"Official CPI vs Negative Sentiment ({TOPIC_LABELS.get(focus_category, focus_category)})")
    ax.set_xlabel("Period")
    ax.set_ylabel("Normalised value")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    # corr_cols = ["official_value", "official_index_value", "complaint_count", "negative_ratio", "avg_sentiment"]
    # corr_cols = [c for c in corr_cols if c in comparison_df.columns]
    # corr = comparison_df[corr_cols].apply(pd.to_numeric, errors="coerce").corr()
    # if sns and not corr.empty:
    #     plt.figure(figsize=(7, 5))
    #     sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", vmin=-1, vmax=1)
    #     plt.title("Correlation Heatmap")
    #     plt.tight_layout()
    #     plt.show()
    # else:
    #     display(corr)

## 12. Year-on-Year Change, Volatility, and Topic Share

These endpoints are in the current frontend contract and are useful for ranking which issues are growing or fluctuating most.

In [ ]:
yoy_df = normalize_columns(rows_df(api_get("/api/categories/yoy-change", {"source_group": DEFAULT_PARAMS["source_group"]})))
volatility_df = normalize_columns(rows_df(api_get("/api/categories/volatility", {"source_group": DEFAULT_PARAMS["source_group"]})))
share_df = normalize_columns(rows_df(api_get("/api/categories/share", {"source_group": DEFAULT_PARAMS["source_group"], "period": DEFAULT_PARAMS["period"]})))

if not yoy_df.empty:
    display(yoy_df.sort_values("yoy_growth", ascending=False))
    yoy_sorted = yoy_df.sort_values("yoy_growth")
    ax = yoy_sorted.plot(kind="barh", x="category_label", y="yoy_growth", legend=False, figsize=(9, 5), color=category_colors(yoy_sorted))
    ax.set_title("Recent 12-Month Category Growth")
    ax.set_xlabel("YoY growth ratio")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

if not volatility_df.empty:
    display(volatility_df.sort_values("monthly_count_std", ascending=False))
    volatility_sorted = volatility_df.sort_values("monthly_count_std")
    ax = volatility_sorted.plot(kind="barh", x="category_label", y="monthly_count_std", legend=False, figsize=(9, 5), color=category_colors(volatility_sorted))
    ax.set_title("Category Volatility")
    ax.set_xlabel("Monthly count standard deviation")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

if not share_df.empty:
    share_pivot = share_df.pivot_table(index="period", columns="category_label", values="percentage", aggfunc="sum", fill_value=0) * 100
    ax = share_pivot.plot(kind="area", stacked=True, figsize=(12, 5), alpha=0.85, color=category_colors_for_labels(share_pivot.columns))
    ax.set_title("Topic Share Over Time")
    ax.set_xlabel("Period")
    ax.set_ylabel("Share (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 13. Keywords

Charts covered: Top Keywords by Category and Word Cloud input table.

In [ ]:
keywords_df = normalize_columns(rows_df(api_get(
    "/api/categories/keywords",
    {"category": "housing,groceries,energy,fuel", "limit": 20, "sample_size": 1000},
)))

if keywords_df.empty:
    show_empty("Keywords")
else:
    display(keywords_df.head(30))
    focus_category = "housing" if "housing" in keywords_df["cost_category"].values else keywords_df["cost_category"].iloc[0]
    focus_keywords = keywords_df[keywords_df["cost_category"] == focus_category].sort_values("frequency", ascending=False).head(15)
    keyword_color = color_for_category(category=focus_category)
    ax = focus_keywords.sort_values("frequency").plot(kind="barh", x="keyword", y="frequency", legend=False, figsize=(8, 5), color=keyword_color)
    ax.set_title(f"Top Keywords: {TOPIC_LABELS.get(focus_category, focus_category)}")
    ax.set_xlabel("Frequency")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

    try:
        from wordcloud import WordCloud
        wc = WordCloud(width=900, height=420, background_color="white", colormap="Set2")
        wc.generate_from_frequencies(dict(zip(focus_keywords["keyword"], focus_keywords["frequency"])))
        plt.figure(figsize=(11, 5))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"Word Cloud: {TOPIC_LABELS.get(focus_category, focus_category)}")
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f"Install wordcloud to render a word cloud. Bar chart above is still available. ({exc})")

## 14. Error Logs

Charts covered: Errors by Function and Recent Error Logs Table.

In [ ]:
errors_df = normalize_columns(rows_df(api_get("/api/logs/errors", {"size": 100})))

if errors_df.empty:
    show_empty("Error Logs", "No recent error rows returned.")
else:
    display(errors_df.head(20))
    if "function_name" in errors_df.columns:
        by_function = errors_df["function_name"].value_counts().sort_values()
        ax = by_function.plot(kind="barh", figsize=(8, 4), color=[NEGATIVE_COLOR] * len(by_function))
        ax.set_title("Errors by Function")
        ax.set_xlabel("Errors")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.show()
    if "error_type" in errors_df.columns:
        display(errors_df["error_type"].value_counts().to_frame("count"))

## 15. Summary for Report

Use the cells below to produce compact numbers for the written analysis. Percentages from the API are 0-1 ratios and are converted to percentages here for presentation.

In [ ]:
summary_rows = []

if "counts_df" in globals() and not counts_df.empty:
    top = counts_df.iloc[0]
    summary_rows.append({"finding": "Most discussed category", "value": f"{top['category_label']} ({int(top['complaint_count']):,} documents)"})

if "sentiment_df" in globals() and not sentiment_df.empty:
    neg = sentiment_df.sort_values("negative_ratio", ascending=False).iloc[0]
    summary_rows.append({"finding": "Highest negative ratio", "value": f"{neg['category_label']} ({neg['negative_ratio'] * 100:.1f}%)"})

if "platforms" in globals() and len(platforms):
    summary_rows.append({"finding": "Largest platform source", "value": f"{platforms.index[0]} ({int(platforms.iloc[0]):,} documents)"})

if "comparison_df" in globals() and not comparison_df.empty:
    overlap = comparison_df["period"].dropna()
    if len(overlap):
        summary_rows.append({"finding": "Official comparison coverage", "value": f"{overlap.min()} to {overlap.max()}"})

pd.DataFrame(summary_rows)